# 03b: Validação da camada Silver

**TC3: Grupo 21 · PosTech FIAP Data Analytics** · Responsável: Caio Bosnic

---

Roda as **54 verificações** da Silver célula a célula, mostrando a evidência de
cada uma, não só `OK` / `FALHA`.

É o mesmo conteúdo de `tests/test_silver_local.py`. As asserções vivem em
`tests/verificacoes_silver.py` e são importadas pelos dois, então notebook e
linha de comando nunca divergem.

| | |
|---|---|
| **Quando usar este notebook** | entender *por que* uma verificação existe, ou investigar uma que falhou |
| **Quando usar o script** | conferir rápido que está tudo verde, antes de subir para o Glue |

> Pré-requisitos (Java e `winutils.exe` no Windows): [`docs/COMO_RODAR_LOCAL.md`](../docs/COMO_RODAR_LOCAL.md)

## 1. Setup

Aponte `PASTA_CSV` para onde estão os três CSVs da Bronze. Os nomes originais
do Gusthavo funcionam, o script acha cada edição pelo ano no fim do nome.

In [ ]:
import os
import sys
import tempfile

def achar_raiz():
    """
    Sobe a partir do diretório atual até achar a raiz do projeto.

    Assim o notebook funciona tanto se você abrir o Jupyter na raiz quanto
    dentro de notebooks/: `os.path.abspath("..")` só funcionava no segundo
    caso e dava ModuleNotFoundError no primeiro.
    """
    p = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(p, "src", "silver", "config_silver.py")):
            return p
        pai = os.path.dirname(p)
        if pai == p:
            break
        p = pai
    raise RuntimeError(
        "Não achei a raiz do projeto a partir de " + os.getcwd() + ".\n"
        "Abra o Jupyter dentro de tech_challenge_3/ (ou de notebooks/)."
    )

RAIZ = achar_raiz()
for sub in ("tests", os.path.join("src", "silver")):
    caminho = os.path.join(RAIZ, sub)
    if caminho not in sys.path:
        sys.path.insert(0, caminho)

# ⬇️ AJUSTE AQUI: pasta com os 3 CSVs da Bronze
PASTA_CSV = r"C:\caminho\da\pasta\com\os\csv"

if not os.path.isdir(PASTA_CSV):
    print(f"[!] PASTA_CSV não existe: {PASTA_CSV}")
else:
    csvs = [f for f in os.listdir(PASTA_CSV) if f.lower().endswith(".csv")]
    print(f"{len(csvs)} CSV(s) em PASTA_CSV: {csvs}")

# Bronze de teste num diretório temporário, o job lê daqui em vez do S3
BASE = tempfile.mkdtemp(prefix="tc3_bronze_")
os.environ["TC3_PATH_BRONZE"] = BASE
os.environ["TC3_PATH_SILVER"] = os.path.join(BASE, "_silver")

print("raiz  :", RAIZ)
print("bronze:", BASE)

In [ ]:
# Se esta célula falhar com ModuleNotFoundError, você pulou a célula anterior, 
# é ela que coloca src/silver e tests no sys.path. Rode na ordem.
import shutil
import subprocess

# O Spark precisa de JVM. Sem Java, o erro que aparece embaixo é confuso
# ("JavaPackage object is not callable"); esta checagem diz o que falta.
if shutil.which("java") is None:
    raise RuntimeError(
        "Java não encontrado no PATH.\n"
        "Instale o JDK 17:  winget install EclipseAdoptium.Temurin.17.JDK\n"
        "Depois FECHE e reabra o Jupyter. Ver docs/COMO_RODAR_LOCAL.md"
    )

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import config_silver as cfg
from job_silver import construir_silver, validar_silver
from verificacoes_silver import (
    LINHAS_ESPERADAS,
    VERIFICACOES,
    Verificador,
    montar_bronze,
    rodar_todas,
)

spark = (
    SparkSession.builder
    .appName("tc3-validacao-silver")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "3g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 2. Montar a Bronze de teste

Converte os CSVs para Parquet, tudo string, no layout que o job espera. É a
Bronze real em miniatura: mesmo formato, mesma decisão de `inferSchema=false`.

In [ ]:
montar_bronze(spark, PASTA_CSV, BASE)

## 3. Construir a Silver

`construir_silver` é a mesma função que o Glue chama. A única diferença entre
rodar aqui e rodar lá são as variáveis de ambiente da célula 1.

In [ ]:
silver = construir_silver(spark).cache()

print(f"{silver.count()} linhas × {len(silver.columns)} colunas")
validar_silver(silver)

## 4. As verificações, uma a uma

Cada célula chama uma função de `verificacoes_silver.py` e mostra a evidência
por trás dela. O objeto `v` acumula os resultados de todas.

O `ctx` carrega o que algumas verificações precisam para reabrir a Bronze crua
e fazer contra-prova.

In [ ]:
v = Verificador()
ctx = {"spark": spark, "path_bronze": BASE}

### 4.1 Contagem por edição

14.002 = 5.293 + 5.215 + 3.494, já sem as 3 duplicatas exatas.

In [ ]:
from verificacoes_silver import verificar_contagens

(silver.groupBy("ano_pesquisa", "edicao")
       .agg(F.count("*").alias("respondentes"))
       .orderBy("ano_pesquisa").show(truncate=False))

verificar_contagens(silver, v, ctx)

### 4.2 Chave

A chave primária muda de nome entre edições (`id` → `token_user`), por isso a
Silver cria a sua: `sha2(ano_pesquisa || id_origem)`. Determinística, 
reprocessar não invalida a Gold.

In [ ]:
from verificacoes_silver import verificar_chave

silver.select("sk_respondente", "ano_pesquisa", "id_origem").show(3, truncate=44)
verificar_chave(silver, v, ctx)

### 4.3 `ano_pesquisa` conferido contra a data de envio

Nenhuma base tem coluna de edição, o ano vem da **origem do arquivo**. Onde
existe data de envio (2024 e 2025), ela tem de cair dentro do ano injetado.
É a contra-prova de que a injeção não trocou os arquivos de lugar.

In [ ]:
from verificacoes_silver import verificar_ano_injetado

(silver.filter(F.col("data_envio").isNotNull())
       .groupBy("ano_pesquisa")
       .agg(F.min("data_envio").alias("primeiro_envio"),
            F.max("data_envio").alias("ultimo_envio"),
            F.count("*").alias("n"))
       .orderBy("ano_pesquisa").show(truncate=False))

verificar_ano_injetado(silver, v, ctx)

### 4.4 Booleanos, o `TRUE`/`FALSE` de 2024-2025

Seis colunas vêm `TRUE`/`FALSE` em 2024-2025 e `0`/`1` nas outras duas. Como a
Bronze é toda string, o `union` engole as duas grafias sem reclamar, e um
`WHERE gestor = '1'` derrubaria os 1.045 gestores de 2024-2025 **sem erro
nenhum**, só um buraco no meio da série.

Se a contagem de gestores tiver valor nos três anos, a normalização pegou.

In [ ]:
from verificacoes_silver import verificar_booleanos

(silver.groupBy("ano_pesquisa")
       .agg(F.sum(F.col("eh_gestor").cast("int")).alias("gestores"),
            F.sum(F.col("empresa_possui_datalake").cast("int")).alias("com_datalake"),
            F.sum(F.col("vive_brasil").cast("int")).alias("vive_no_brasil"))
       .orderBy("ano_pesquisa").show())

verificar_booleanos(silver, v, ctx)

### 4.5 Nenhuma coluna 100% nula

O sintoma clássico de de-para apontando para um nome que não existe: a coluna
sai, o job não reclama, e o dado está vazio.

Foi essa verificação que pegou o `empresa_passou_layoff` tratado como booleano
quando ele tem três respostas.

In [ ]:
from verificacoes_silver import verificar_colunas_nao_vazias
verificar_colunas_nao_vazias(silver, v, ctx)

### 4.6 Layoff: 3 respostas viram 2 flags

`empresa_passou_layoff` não é booleana: não houve / houve e não me afetou /
houve e fui afetado. A Silver guarda a categoria e deriva dois flags, porque
"houve layoff na empresa" e "o layoff me atingiu" são perguntas diferentes.

In [ ]:
from verificacoes_silver import verificar_layoff

(silver.filter(F.col("empresa_passou_layoff").isNotNull())
       .groupBy("houve_layoff", "fui_afetado_layoff")
       .agg(F.count("*").alias("n")).orderBy("houve_layoff", "fui_afetado_layoff").show())

verificar_layoff(silver, v, ctx)

### 4.7 `uf_moradia`, o achado do drift semântico

O mais perigoso da base. Em 2023-2024 e 2024-2025 a coluna `uf` é **onde a
pessoa mora**; em 2025-2026 é **onde ela nasceu**. Não gera erro, não gera
nulo, só produz número errado na análise regional.

A Silver ignora `uf` e deriva a sigla de `estado`. A célula abaixo mostra o
problema na Bronze crua e a correção na Silver.

In [ ]:
from verificacoes_silver import verificar_uf_moradia

sigla = lambda c: F.regexp_extract(F.col(c), r"\(([A-Z]{2})\)", 1)

print("Na BRONZE crua: `uf` bate com qual coluna?\n")
print("         estado (mora)   estado_origem (nasceu)")
for ano, meta in cfg.EDICOES.items():
    cru = spark.read.parquet(os.path.join(BASE, meta["tabela_bronze"]))
    linha = f"  {ano}  "
    for alvo in ("estado", "estado_origem"):
        if alvo not in cru.columns or "uf" not in cru.columns:
            linha += f"{'--':>15s}"
            continue
        linha += f"{cru.filter(F.col('uf') == sigla(alvo)).count() / cru.count():>14.1%} "
    print(linha)

print("\nNa SILVER, uf_moradia é sempre onde mora:")
verificar_uf_moradia(silver, v, ctx)

### 4.8 Salário, faixa vira ponto médio

`salario_medio_mensal` existe para permitir média e mediana na Gold. Duas
ressalvas: a faixa aberta do topo usa 40.001 (subestima, viés conservador) e o
número **só faz sentido reportado junto da faixa**.

Aqui também entram os dois typos da origem (`R$_101` e `R$_3000`), que viravam
categoria órfã no `GROUP BY`.

In [ ]:
from verificacoes_silver import verificar_salario

(silver.filter(F.col("faixa_salarial").isNotNull())
       .groupBy("faixa_salarial", "salario_medio_mensal")
       .agg(F.count("*").alias("n"))
       .orderBy("salario_medio_mensal").show(20, truncate=38))

verificar_salario(silver, v, ctx)

### 4.9 Comparabilidade de série

1.282 linhas têm valor que não existe nas três edições: `Especialista/Staff+`
na senioridade (só 2025), arquiteto como cargo separado, `Ciencia_de_Dados/IA`
na formação.

Sem a marcação, categoria nova de questionário vira "crescimento" no gráfico.

In [ ]:
from verificacoes_silver import verificar_comparabilidade

(silver.filter(~F.col("serie_comparavel"))
       .groupBy("ano_pesquisa", "nivel_senioridade", "cargo_atual")
       .agg(F.count("*").alias("n")).orderBy(F.desc("n")).show(8, truncate=44))

verificar_comparabilidade(silver, v, ctx)

### 4.10 Pergunta ausente vira NULL, não zero

`estado_origem` não existe em 2023-2024; `llms_bom_resultado` só existe em
2025-2026. Materializar zero em vez de NULL faria a série parecer queda a
zero, quando na verdade a pergunta não foi feita.

In [ ]:
from verificacoes_silver import verificar_ausentes_viram_nulo
verificar_ausentes_viram_nulo(silver, v, ctx)

### 4.11 Múltipla escolha, contra-prova com a Bronze crua

Pega uma linha real, volta na Bronze e confere se a lista consolidada tem
exatamente as opções marcadas com `1` na origem. É o teste que garante que a
consolidação de ~320 binários em 17 grupos não inventou nem perdeu opção.

In [ ]:
from verificacoes_silver import verificar_multipla_escolha_contra_origem

exemplo = (silver.filter((F.col("ano_pesquisa") == 2025) & (F.col("qtd_linguagens") >= 3))
                 .select("id_origem", "cargo_atual", "linguagens", "qtd_linguagens").first())
print("Silver:", exemplo["linguagens"], f"({exemplo['qtd_linguagens']} opções)")

cru = spark.read.parquet(os.path.join(BASE, cfg.EDICOES[2025]["tabela_bronze"]))
origem = cru.filter(F.col("token_user") == exemplo["id_origem"]).first()
marcadas = [c for c in cfg.GRUPOS_MULTIPLA_ESCOLHA["linguagens"][2025]
            if c in cru.columns and origem[c] == "1"]
print("Bronze:", ", ".join(sorted(marcadas)), f"({len(marcadas)} colunas com '1')")

verificar_multipla_escolha_contra_origem(silver, v, ctx)

### 4.12 Os 17 grupos, presença e coerência

Todo grupo virou par (lista + contagem), tem conteúdo, e as duas colunas ficam
nulas **juntas**.

⚠️ **NULL não é zero.** Muitos blocos só aparecem para parte da base, 
`atividades_cientista_dados` tem 2.010 respondentes de 14.002. Quem não viu a
pergunta fica NULL; quem viu e não marcou fica `''` e `0`. Calcular percentual
sobre 14.002 derruba o número pela metade.

In [ ]:
from verificacoes_silver import verificar_grupos

import pandas as pd
resumo = []
for g in sorted(cfg.GRUPOS_MULTIPLA_ESCOLHA):
    tamanhos = "/".join(str(len(cfg.GRUPOS_MULTIPLA_ESCOLHA[g].get(a, []))) for a in (2023, 2024, 2025))
    resumo.append({"grupo": g, "opções (23/24/25)": tamanhos})
display(pd.DataFrame(resumo))

verificar_grupos(silver, v, ctx)

## 5. Resultado

Se você rodou as células acima na ordem, `v` já tem as 54. Para rodar tudo de
uma vez sem passar célula a célula, use `rodar_todas`.

In [ ]:
display(v.tabela())
print(v.resumo())

In [ ]:
# Bateria completa do zero, sem a evidência intermediária
# v2 = rodar_todas(silver, ctx, silencioso=True)
# display(v2.tabela())
# print(v2.resumo())

## 6. Encerrar

Libera a JVM e apaga a Bronze temporária. Se quiser explorar a Silver depois
sem remontar tudo, use `scripts/preparar_bronze_local.py`, que grava numa
pasta que fica.

In [ ]:
import shutil

spark.stop()
shutil.rmtree(BASE, ignore_errors=True)
print("encerrado")